# Beyond GDP: Human Development and Capability Analysis

## Notebook 4 — Statistical Analysis

In the chart stage we could see which things move together. But our eyes can fool
us. In this notebook we use numbers to check those patterns properly.

We do two things:
1. Correlation: a single number that tells how strongly two things move together.
2. Regression: a way to find out which factors explain HDI the most.

One rule we follow: even if two things move together, it does not prove one causes
the other. We keep that in mind the whole time.

## 1. Setup

We load the cleaned master dataset and the libraries we need. We use scipy for
correlation and statsmodels for regression. Both are standard tools for this kind
of analysis.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from pathlib import Path

master_path = Path(r"D:\beyond-gdp-capability-analysis\data\processed\master_dataset.csv")
df = pd.read_csv(master_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (130, 18)
Columns: ['country', 'code', 'hdi', 'life_expectancy', 'expected_schooling', 'mean_schooling', 'gni_per_capita', 'happiness_score', 'social_support_contrib', 'freedom_contrib', 'generosity_contrib', 'corruption_contrib', 'gdp_per_capita', 'health_expenditure', 'unemployment', 'inflation', 'population', 'cpi_score']


## 2. Correlation with HDI

Correlation is a single number between -1 and +1 that tells how strongly two things
move together. We check how each factor relates to HDI.

We use two methods. Pearson looks for a straight-line relationship. Spearman only
looks at the order (which country is higher or lower), so it works even when the
relationship is a curve. We saw earlier that GDP and HDI form a curve, so showing
both is useful.

In [2]:
factors = ["gdp_per_capita", "gni_per_capita", "life_expectancy",
           "mean_schooling", "happiness_score", "health_expenditure",
           "unemployment", "inflation", "cpi_score"]

results = []
for col in factors:
    pearson = df["hdi"].corr(df[col], method="pearson")
    spearman = df["hdi"].corr(df[col], method="spearman")
    results.append([col, round(pearson, 2), round(spearman, 2)])

corr_table = pd.DataFrame(results, columns=["factor", "pearson", "spearman"])
corr_table = corr_table.sort_values("pearson", ascending=False).reset_index(drop=True)

print(corr_table.to_string(index=False))

            factor  pearson  spearman
   life_expectancy     0.93      0.93
    mean_schooling     0.93      0.91
    gni_per_capita     0.83      0.98
   happiness_score     0.79      0.84
         cpi_score     0.72      0.77
    gdp_per_capita     0.71      0.98
health_expenditure     0.47      0.56
      unemployment     0.02      0.11
         inflation    -0.06     -0.16


**What this table shows:**

The number tells how strongly each factor moves with HDI. Closer to 1 means a
stronger link.

Strongest links:
- Life expectancy (0.93) and schooling (0.93). This is expected, because HDI is
  partly built from health and education.
- Happiness (0.79) and corruption score (0.72) are also strong, even though they
  are not part of HDI's formula. So well-being and clean governance go with
  development too.

The most interesting result is GDP:
- Pearson for GDP is 0.71, but Spearman is 0.98.
- Why the big gap? Pearson checks for a straight line, and GDP vs HDI is not a
  straight line, it is a curve (money helps a lot when poor, little when rich).
  Spearman only checks the order of countries, so it ignores the curve and sees a
  very strong link.
- In plain words: richer countries are almost always higher on HDI (order is very
  consistent), but the amount of extra HDI you get per extra dollar shrinks as
  countries get rich. That single difference between 0.71 and 0.98 captures the
  whole "money has limits" idea in one place.

Weakest links:
- Unemployment (0.02) and inflation (-0.06) have almost no link with HDI here. These
  short-term economic numbers do not explain long-term development.

So health, education, happiness and governance all connect strongly to development,
while raw yearly economic numbers like inflation do not.

## 3. Regression: what explains HDI beyond its own parts

Correlation looks at one factor at a time. Regression looks at several factors
together and tells which ones matter most when the others are also considered.

Important point: HDI is built from health, education and income, so using those to
predict HDI would be circular. Instead we use factors that are not inside HDI:
GDP per capita, health expenditure, unemployment, happiness, freedom and the
corruption score. The question becomes: beyond its own parts, how well do these
outside factors explain HDI, and which ones matter most.

We also note a limitation up front: some of these factors overlap with each other
(for example GDP and happiness are related), which can make individual effects
harder to separate. We read the results with that in mind.

In [3]:
predictors = ["gdp_per_capita", "health_expenditure", "unemployment",
              "happiness_score", "freedom_contrib", "cpi_score"]

X = df[predictors]
y = df["hdi"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    hdi   R-squared:                       0.736
Model:                            OLS   Adj. R-squared:                  0.724
Method:                 Least Squares   F-statistic:                     57.28
Date:                Thu, 25 Jun 2026   Prob (F-statistic):           2.64e-33
Time:                        04:25:43   Log-Likelihood:                 146.96
No. Observations:                 130   AIC:                            -279.9
Df Residuals:                     123   BIC:                            -259.8
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.2137      0

**What this regression shows:**

We tried to explain HDI using six outside factors (not HDI's own parts).

1. R-squared = 0.736
   This means these six factors together explain about 74% of why HDI differs
   between countries. That is a strong result. So even without using HDI's own
   ingredients, outside factors explain most of the differences in development.

2. Which factors matter (look at the P>|t| column; below 0.05 means it is a real
   effect, not luck):
   - happiness_score: p = 0.000, strong and real. Higher happiness goes with higher
     HDI even after accounting for the rest.
   - cpi_score (clean governance): p = 0.008, real and positive.
   - unemployment: p = 0.004, real (small positive effect here, which is a bit odd;
     noted as a limitation below).
   - gdp_per_capita: p = 0.119, NOT significant once the other factors are included.
   - health_expenditure: p = 0.371, not significant here.
   - freedom_contrib: p = 0.287, not significant here.

3. The big takeaway:
   This is the project's main point, now in numbers. When we hold everything
   together, GDP is NOT a significant explainer of HDI, but happiness and clean
   governance ARE. Money alone does not stand out once human and institutional
   factors are in the picture.

Limitations (we state these honestly):
- The note at the bottom warns of multicollinearity, meaning some factors overlap
  (for example GDP, happiness and governance are themselves related). This makes it
  hard to fully trust each single coefficient, though the overall fit is still
  meaningful.
- The unemployment result is positive, which is unexpected. With overlap in the
  data, single effects like this can be unstable, so we do not over-read it.
- This is one year of data and association only. It does not prove cause and effect.

## 4. Hypothesis test: are high-HDI countries happier?

We split countries into two groups: high HDI (0.8 and above) and lower HDI (below
0.8). Then we test whether the difference in their average happiness is real or
could just be chance.

We use a t-test. It gives a p-value:
- p-value below 0.05 means the difference is statistically real.
- p-value above 0.05 means the difference could be due to chance.

We state the idea being tested before reading the result.

In [4]:
high_hdi = df[df["hdi"] >= 0.8]["happiness_score"]
low_hdi = df[df["hdi"] < 0.8]["happiness_score"]

print("High HDI group: countries =", len(high_hdi),
      ", average happiness =", round(high_hdi.mean(), 2))
print("Low HDI group:  countries =", len(low_hdi),
      ", average happiness =", round(low_hdi.mean(), 2))

t_stat, p_value = stats.ttest_ind(high_hdi, low_hdi)

print("\nt-statistic:", round(t_stat, 3))
print("p-value:", p_value)

High HDI group: countries = 59 , average happiness = 6.46
Low HDI group:  countries = 71 , average happiness = 4.88

t-statistic: 10.602
p-value: 3.096809108865995e-19


## Statistics summary

We used numbers to test the patterns we saw in the charts. Three things came out:

1. What goes together with development (HDI):
   Health, education, happiness and clean government all move closely with HDI.
   Where these are high, HDI is usually high too.

2. The most important finding:
   When we looked at all factors together, money (GDP) alone could not explain a
   country's development. But happiness and clean government clearly could. So a
   country is not developed just because it is rich, other things matter as much.

3. Are richer-development countries happier?
   Yes. Countries with high HDI had an average happiness of 6.46, while lower-HDI
   countries averaged 4.88. The test showed this gap is real, not just luck.

What we are careful about:
   These results show that things move together, not that one causes the other.
   The data is for one year (2023), and some factors overlap with each other, so we
   do not over-read any single number. Still, the overall message is clear: GDP by
   itself is not enough to explain how developed or how happy a country is.